# 02 · Scaled dot-product attention
*La fórmula desde cero: producto punto, raíz de d_k, softmax por filas y máscara*

Notebook del episodio 2 de **Attention is all you need** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/attention-is-all-you-need/scaled-dot-product)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** 3 tokens a mano (S, A, out) → Var(q·k) = d_k (nota 4) → sin escala, d_k = 512: peso máx 0,95 y gradiente 11× menor → derivada del softmax a_i(δ_ij − a_j) → función attention reutilizable → dim=0 no avisa → máscara causal: cambiar el último token no toca los anteriores → = F.scaled_dot_product_attention (is_causal y attn_mask negada)

**Cómo correrlo:** con el entorno de `~/Movies/serie-attention/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "attention"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Attention is all you need. Episodio dos: scaled dot-product attention. En el episodio uno vimos el problema: las redes recurrentes leen de a un token y todo pasa por un vector. Hoy, la pieza que las reemplaza, escrita desde cero en PyTorch, con cada decisión explicada, y verificada contra la función de la librería.*

### De dónde venimos

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\Big(\frac{Q K^\top}{\sqrt{d_k}}\Big)\, V
$$
<small>la ecuación (1) del paper</small>

$$
A\, V:\; (T \times T)(T \times d_v)
$$
<small>cada salida: promedio de las filas de $V$</small>

En Alineación, episodio uno, apareció esta fórmula, y en Álgebra lineal para LLMs, episodio uno, medimos por qué se divide por raíz de d. Q, K y V son matrices con una fila por token. En el dibujo, la fila i de Q por la columna j de K transpuesta llena la celda i j de una tabla de T por T. Se divide por raíz de d sub k y pasa por un softmax. Esa tabla, A, multiplica a V: la fila i de A pesa las filas de V y da la salida del token i.

> **Vaswani et al., 2017 — Attention Is All You Need** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **query + key-value → output** · «An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors.»

Así la define el paper, en la sección tres punto dos. Una función de atención toma una query y un conjunto de pares key, value, y devuelve una salida. Todo son vectores. La salida es una suma pesada de los values, y el peso de cada value sale de comparar la query con su key. Query, key y value: así se dicen en todo el código y en todos los papers, así que los vamos a usar en inglés.

## Parte 1 · Query, key, value

Primero la intuición, después tres tokens con números redondos que se pueden seguir a mano.

### Un diccionario con búsqueda suave

- Query $q$: lo que el token está buscando
- Key $k$: la etiqueta con la que cada token se ofrece
- Value $v$: lo que el token entrega si lo eligen
- No elige una key: promedia todos los values según el parecido

Pensá en un diccionario de Python. Le pasás una clave, la query: lo que buscás. Cada entrada tiene su key, la etiqueta con la que se ofrece. Y su value, lo que devuelve. La diferencia: acá no hay coincidencia exacta. Cada key se compara con la query con un producto punto, grande cuando los vectores apuntan para el mismo lado, y la salida es un promedio de todos los values pesado por ese parecido. Una búsqueda suave, y por eso se puede derivar y entrenar.

### Tres tokens, a mano

Tres tokens, y d sub k igual a dos. La query del token cero es dos, cero; la del uno, cero, dos; la del dos, uno, uno. Las keys: uno, cero; cero, uno; y uno, uno. Los values: diez, cero; cero, diez; y cinco, cinco. Los scores: Q por K transpuesta, dividido por raíz de dos. Softmax por filas. Y A por V. La fila cero de S: uno coma cuarenta y uno, cero, uno coma cuarenta y uno. Los pesos: cero coma cuarenta y cinco, cero coma once, cero coma cuarenta y cinco. Y la salida del token cero, seis coma sesenta y nueve, tres coma treinta y uno: tira hacia el primer value.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_ejemplo.py
__file__ = 'snippets/ep02_ejemplo.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.set_printoptions(precision=2, sci_mode=False)

In [3]:
# 3 tokens, d_k = 2: números redondos para seguir a mano
Q = torch.tensor([[2., 0.], [0., 2.], [1., 1.]])
K = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
V = torch.tensor([[10., 0.], [0., 10.], [5., 5.]])

S = Q @ K.T / 2 ** 0.5       # [3, 3]: q_i · k_j / √d_k
A = torch.softmax(S, dim=-1) # cada fila suma 1
out = A @ V                  # [3, 2]: promedio pesado de V
print("S =\n", S)
print("A =\n", A)
print("out =\n", out)

S =
 tensor([[1.41, 0.00, 1.41],
        [0.00, 1.41, 1.41],
        [0.71, 0.71, 1.41]])
A =
 tensor([[0.45, 0.11, 0.45],
        [0.11, 0.45, 0.45],
        [0.25, 0.25, 0.50]])
out =
 tensor([[6.69, 3.31],
        [3.31, 6.69],
        [5.00, 5.00]])


### La fila 0, paso por paso

$$
S_{00} = \frac{q_0 \cdot k_0}{\sqrt{2}} = \frac{2 \cdot 1 + 0 \cdot 0}{1{,}41} = 1{,}41
$$
<small>query 0 contra key 0</small>

$$
A_{0j} = \frac{e^{S_{0j}}}{\sum_m e^{S_{0m}}}
$$
<small>softmax de la fila: pesos que suman 1</small>

$$
\text{out}_0 = 0{,}45\, v_0 + 0{,}11\, v_1 + 0{,}45\, v_2 = (6{,}69;\ 3{,}31)
$$
<small>el promedio pesado de los values</small>

La fila cero, despacio. El score contra la key cero: multiplicar coordenada a coordenada y sumar. Dos por uno más cero por cero, dos, dividido por raíz de dos: uno coma cuarenta y uno. En el dibujo, los dos productos y su suma. El softmax de la fila: e a la cada score, dividido por la suma de todos. La exponencial vuelve todo positivo y la división hace que sume uno. En el dibujo, scores a la izquierda, pesos a la derecha. La salida: cero coma cuarenta y cinco del primer value, cero coma once del segundo y cero coma cuarenta y cinco del tercero. En el dibujo, cada value con su peso.

### La matriz A del ejemplo (salida real)

La matriz A entera. El token cero mira a las keys cero y dos, que tienen un uno en la primera coordenada, como su query. El uno, simétrico: a la uno y a la dos. Y el dos, que busca uno, uno, le da la mitad a la key dos, igual a su query, y reparte el resto. Cada fila suma uno y responde: a quién le presta atención este token.

## Parte 2 · Por qué raíz de d sub k

La palabra scaled del nombre es la división por raíz de d sub k. Sin ella, el modelo aprende mal. Veamos por qué.

### El producto punto crece con la dimensión

$$
q \cdot k = \textstyle\sum_{i=1}^{d_k} q_i\, k_i
$$
<small>una suma de $d_k$ productos</small>

$$
\mathrm{Var}(q \cdot k) = \textstyle\sum_i \mathrm{Var}(q_i k_i) = d_k
$$
<small>si cada $q_i$, $k_i$ tiene media 0 y varianza 1</small>

$$
\mathrm{Var}\Big(\frac{q \cdot k}{\sqrt{d_k}}\Big) = \frac{d_k}{d_k} = 1
$$
<small>dividir por el desvío lo deja en 1</small>

El argumento está en la nota cuatro del paper. El producto punto es una suma de d sub k productos: en el dibujo, cuatro términos, unos positivos y otros negativos. La varianza mide cuánto se aleja algo de cero, en promedio, al cuadrado. Si cada coordenada de q y de k es independiente, con media cero y varianza uno, cada producto tiene varianza uno, y las varianzas de términos independientes se suman: d sub k. El desvío, la raíz, es raíz de d sub k: dos, ocho, veintidós coma seis. Dividiendo por raíz de d sub k, la varianza vuelve a uno. Eso ya lo medimos en Álgebra; lo nuevo es qué le pasa al softmax.

### Softmax saturado, medido

Mil queries al azar, cada una contra ocho keys. Los ocho scores de cada query. Con escala o sin escala. Backward desde el peso del primer token: cuánto gradiente vuelve a los scores. Y el peso más grande de cada fila, en promedio. Sin escalar, el peso máximo sube con d sub k: cero coma cincuenta y tres, cero coma ochenta y siete, cero coma noventa y cinco. Casi todo a un token. Y el gradiente cae de diecinueve milésimas a dos. Escalando, el peso máximo queda en cero coma treinta y cinco o treinta y seis y el gradiente en veinticuatro milésimas, para cualquier d sub k.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_saturacion.py
__file__ = 'snippets/ep02_saturacion.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)

In [5]:
def medir(d_k, escalar):
    q = torch.randn(1000, 1, d_k)            # 1000 consultas
    k = torch.randn(1000, 8, d_k)            # 8 claves cada una
    s = (q * k).sum(-1)                      # [1000, 8] productos punto
    if escalar:
        s = s / d_k ** 0.5
    s.requires_grad_(True)
    a = s.softmax(-1)
    a[:, 0].sum().backward()                 # ∂ peso del token 0 / ∂ scores
    return a.max(-1).values.mean(), s.grad.abs().mean()

for d_k in [4, 64, 512]:
    for esc in (False, True):
        pmax, g = medir(d_k, esc)
        print(f"d_k={d_k:3} {'÷√d_k' if esc else 'crudo'}:"
              f" peso máx {pmax:.2f}  |grad| {g:.4f}")

d_k=  4 crudo: peso máx 0.53  |grad| 0.0190
d_k=  4 ÷√d_k: peso máx 0.35  |grad| 0.0241
d_k= 64 crudo: peso máx 0.87  |grad| 0.0060
d_k= 64 ÷√d_k: peso máx 0.36  |grad| 0.0240
d_k=512 crudo: peso máx 0.95  |grad| 0.0022
d_k=512 ÷√d_k: peso máx 0.36  |grad| 0.0243


### Gradiente que vuelve a los scores (salida real, ×10⁻³)

El gradiente, en barras: milésimas. Con d sub k igual a cuatro, escalar casi no cambia nada. Diecinueve contra veinticuatro. Con sesenta y cuatro, sin escalar baja a seis. Escalado, veinticuatro. Con quinientos doce, dos coma dos: once veces menos. Escalado, sigue en veinticuatro. Un gradiente once veces más chico es una señal de aprendizaje once veces más débil para Q y K.

- **d=4 crudo**: 19,0
- **d=4 ÷√d**: 24,1
- **d=64 crudo**: 6,0
- **d=64 ÷√d**: 24,0
- **d=512 crudo**: 2,2
- **d=512 ÷√d**: 24,3

### Por qué el gradiente se apaga

$$
\frac{\partial a_i}{\partial s_j} = a_i\,(\delta_{ij} - a_j)
$$
<small>la derivada del softmax; $\delta_{ij} = 1$ si $i = j$, si no 0</small>

$$
a \approx (1, 0, \dots, 0) \;\Rightarrow\; a_i(\delta_{ij} - a_j) \approx 0
$$
<small>saturado: ningún score mueve nada</small>

$$
\mathrm{softmax}\Big(\frac{s}{\sqrt{512}}\Big)
$$
<small>los mismos scores, escalados</small>

Por qué pasa. a es el vector de pesos y s el de scores. La derivada del peso i respecto del score j es a sub i por delta i j menos a sub j; delta i j vale uno si i es igual a j, y cero si no. En el dibujo, seis scores crudos del tamaño que da d sub k igual a quinientos doce: uno se lleva el noventa y nueve coma nueve por ciento. Con un peso casi uno y el resto casi cero, cada término de la derivada tiene un factor casi cero: en el dibujo, todo aplastado contra el piso. Divididos por raíz de quinientos doce, los mismos scores dan pesos repartidos, y la derivada vive.

> **Vaswani et al., 2017 — Attention Is All You Need, nota 4** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **Var(q · k) = d_k** · «We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients.»

El paper lo dice casi igual. Sospechan que para d sub k grande los productos punto crecen, y empujan al softmax a regiones donde tiene gradientes extremadamente chicos. Y en la nota cuatro, el argumento que acabamos de ver: con coordenadas independientes de media cero y varianza uno, q punto k tiene varianza d sub k. Fijate que dicen sospechamos: es un argumento razonable, no un teorema. Nuestro experimento lo respalda.

## Parte 3 · Softmax por filas, y la máscara

Dos detalles de implementación que, si salen mal, no tiran error: devuelven números equivocados.

### La función entera

Nuestra attention, la que reusamos toda la serie. d sub k sale de la última dimensión de Q. Transpose de menos dos y menos uno da vuelta las dos últimas dimensiones: así funciona igual con un batch de frases adelante. Las posiciones prohibidas se llenan con menos infinito. Softmax sobre la última dimensión: por filas. Devolvemos la salida y A, para mirarla. Q, K y V de cuatro por ocho; A de cuatro por cuatro; cada fila suma uno.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_atencion.py
__file__ = 'snippets/ep02_atencion.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
torch.manual_seed(0)
T, d_k, d_v = 4, 8, 8
Q, K, V = torch.randn(T, d_k), torch.randn(T, d_k), torch.randn(T, d_v)

In [7]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # [.., T_q, T_k]
    if mask is not None:                           # True = prohibido
        S = S.masked_fill(mask, float("-inf"))
    A = torch.softmax(S, dim=-1)                   # por filas
    return A @ V, A                                # [.., T_q, d_v]

out, A = attention(Q, K, V)
print("Q", tuple(Q.shape), "K", tuple(K.shape), "V", tuple(V.shape))
print("A", tuple(A.shape), " out", tuple(out.shape))
print("suma de cada fila de A:", A.sum(-1).numpy().round(3))

Q (4, 8) K (4, 8) V (4, 8)
A (4, 4)  out (4, 8)
suma de cada fila de A: [1. 1. 1. 1.]


### dim=-1 contra dim=0

Por qué insistir con por filas. Con dim igual a menos uno, cada fila se normaliza: cada query reparte su atención. Con dim igual a cero, se normaliza cada columna: cada key reparte su influencia entre las queries. Eso no es atención. Y no explota nada. Con dim menos uno, cada fila suma uno. Con dim cero, las filas suman cero coma noventa y cuatro, cero coma treinta y tres, y uno coma setenta y tres; las que suman uno son las columnas. Mismas formas, ningún error, y un modelo que entrena peor sin decirte por qué.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_filas.py
__file__ = 'snippets/ep02_filas.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)
S = torch.randn(3, 3) * 2

In [9]:
bien = torch.softmax(S, dim=-1)   # normaliza cada fila
mal = torch.softmax(S, dim=0)     # normaliza cada columna

print("dim=-1, suma por fila:   ", bien.sum(-1).numpy().round(3))
print("dim=0,  suma por fila:   ", mal.sum(-1).numpy().round(3))
print("dim=0,  suma por columna:", mal.sum(0).numpy().round(3))

dim=-1, suma por fila:    [1. 1. 1.]
dim=0,  suma por fila:    [0.936 0.33  1.734]
dim=0,  suma por columna: [1. 1. 1.]


### La máscara causal

Al generar texto, un token no puede mirar a los que vienen después. triu arma una matriz con True arriba de la diagonal: j mayor que i, el futuro. Sumar M, menos infinito ahí y cero en el resto, es lo mismo que masked fill. Self-attention: los mismos vectores X hacen de query, key y value; en un modelo de verdad pasan antes por tres matrices. e a la menos infinito es cero: peso exactamente cero. Para probarlo, cambiamos el último token. A es triangular. Las salidas de los tokens cero a dos no cambiaron: True. Solo cambió la del token tres. Cada salida depende solo de su pasado.

**Cada línea, su término:**

- `mask = torch.triu` → $M_{ij} = \begin{cases} -\infty & j > i \\ 0 & j \le i \end{cases}$ — prohibido mirar el futuro
- `out, A = attention(X, X, X, mask)` → $A = \mathrm{softmax}\Big(\frac{X X^\top}{\sqrt{d}} + M\Big)$ — self-attention: $Q = K = V = X$
- `X2[3] = torch.randn(d)` → $e^{-\infty} = 0$ — lo enmascarado pesa exactamente cero
- `tokens 0–2 iguales` → $\text{out}_i = f(x_0, \dots, x_i)$ — cada salida solo depende del pasado

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_mascara.py
__file__ = 'snippets/ep02_mascara.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
torch.manual_seed(0)
torch.set_printoptions(precision=2)


def attention(Q, K, V, mask=None):
    S = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        S = S.masked_fill(mask, float("-inf"))
    A = torch.softmax(S, dim=-1)
    return A @ V, A


T, d = 4, 8
X = torch.randn(T, d)

In [11]:
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
out, A = attention(X, X, X, mask)          # self-attention: Q = K = V = X
print(A)

X2 = X.clone()
X2[3] = torch.randn(d)                     # cambiamos el ÚLTIMO token
out2, _ = attention(X2, X2, X2, mask)
print("tokens 0–2 iguales:", torch.allclose(out[:3], out2[:3]))
print("token 3 igual:     ", torch.allclose(out[3], out2[3]))

tensor([[1.00, 0.00, 0.00, 0.00],
        [0.27, 0.73, 0.00, 0.00],
        [0.01, 0.01, 0.98, 0.00],
        [0.17, 0.23, 0.01, 0.58]])
tokens 0–2 iguales: True
token 3 igual:      False


### A con máscara (salida real)

La matriz A de ese ejemplo. El token cero solo se puede mirar a sí mismo: peso uno. El uno reparte entre el cero y él. El dos casi todo para sí: cero coma noventa y ocho, porque sin proyecciones, el producto punto de un vector consigo mismo suele ser el más grande. Y el tres reparte entre los cuatro. Esta es la máscara que el decoder del transformer usa al generar; la volvemos a ver en el episodio cinco.

## Parte 4 · Contra PyTorch

Una implementación propia vale si coincide con una de referencia. PyTorch trae la suya.

### F.scaled_dot_product_attention

Un batch de dos frases, seis tokens, d sub k sesenta y cuatro, como cada head del paper. La máscara: True donde está prohibido. Nuestra función. La de PyTorch con is causal: arma la máscara causal sola. Y con attn mask. Ojo: en PyTorch, una máscara booleana marca True lo permitido, al revés que la nuestra; por eso va negada. Si te olvidás, mira el futuro sin dar error. Las dos coinciden con la nuestra. Diferencia máxima: dos por diez a la menos siete, redondeo de punto flotante.

In [12]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep02_verificar.py
__file__ = 'snippets/ep02_verificar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
import torch.nn.functional as F
torch.manual_seed(0)


def attention(Q, K, V, mask=None):
    S = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        S = S.masked_fill(mask, float("-inf"))
    A = torch.softmax(S, dim=-1)
    return A @ V, A

In [13]:
B, T, d_k = 2, 6, 64                       # batch de 2 frases
Q, K, V = (torch.randn(B, T, d_k) for _ in range(3))
prohibido = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)

nuestro, _ = attention(Q, K, V, prohibido)
torch_a = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
torch_b = F.scaled_dot_product_attention(Q, K, V, attn_mask=~prohibido)

print("vs is_causal=True:", torch.allclose(nuestro, torch_a, atol=1e-6))
print("vs attn_mask:     ", torch.allclose(nuestro, torch_b, atol=1e-6))
print("diferencia máxima:", f"{(nuestro - torch_a).abs().max():.1e}")

vs is_causal=True: True
vs attn_mask:      True
diferencia máxima: 2.4e-07


### Lo que la fórmula no tiene todavía

- Pesos propios: $W^Q, W^K, W^V$ llegan con multi-head (ep. 3)
- Orden: sin positional encoding, la posición no existe (ep. 4)
- Costo: $A$ es $T \times T$; el paper lo cuenta como $O(n^2 \cdot d)$
- Una sola $A$ por capa: un token, un único reparto

Lo que esta fórmula todavía no tiene. Pesos que aprender: hoy Q, K y V vinieron dados. En el episodio tres aparecen las matrices que los proyectan, y las heads. Orden: si permutás los tokens, la salida se permuta igual. Lo demostramos en el episodio cuatro. Y tiene un costo: A tiene T por T celdas. La tabla uno del paper lo cuenta como n al cuadrado por d por capa. Y con una sola A, cada token tiene un único reparto de atención. Por eso el paper usa varias.

## Resumiendo

- Query, key, value: un diccionario con búsqueda suave
- $S = QK^\top/\sqrt{d_k}$: un producto punto por par de tokens
- Sin $\sqrt{d_k}$: $d_k = 512$ → peso máx 0,95 y gradiente 11× menor
- Softmax con dim=-1: cada fila suma 1; dim=0 no avisa
- Máscara: $-\infty$ en el futuro → peso 0 exacto
- Próximo: multi-head attention, con las heads de Gemma